[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elchapa24/tpi-analisis-de-datos/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb)

# Trabajo Práctico Integrador - Análisis de Datos
## Telco Customer Churn

### A. Fundamentos, Ingesta y Limpieza de Datos

#### 1. Definición del problema de negocio
La empresa de telecomunicaciones enfrenta una pérdida constante de clientes (Churn). El objetivo de este análisis es identificar qué factores sociodemográficos y de contratación están asociados a la cancelación del servicio, para diseñar estrategias de retención efectivas.

**Variable de interés principal:** `Churn` (Cancelación del servicio: Yes/No).

**Hipótesis analíticas iniciales:**
1. Los clientes con contratos mes a mes (*Month-to-month*) tienen una tasa de cancelación significativamente mayor.
2. La antigüedad del cliente (*tenure*) está inversamente relacionada con el Churn.
3. El método de pago influye en la retención.

#### 2. Clasificación del Dataset y Variables
- **Estructura:** El dataset es **estructurado** (tabular).
- **Clasificación teórica:**
  - *Cualitativas Nominales:* `gender`, `PaymentMethod`, `InternetService`.
  - *Cualitativas Ordinales:* `Contract` (Month-to-month < One year < Two year).
  - *Cuantitativas Discretas:* `SeniorCitizen` (0/1), `tenure` (meses).
  - *Cuantitativas Continuas:* `MonthlyCharges`, `TotalCharges`.
- **Contraste con Pandas:** Pandas leerá `TotalCharges` como `object` (texto) por culpa de espacios vacíos. Esto se corregirá en la limpieza.

In [1]:
import pandas as pd
import numpy as np

# 1. CARGA DE DATOS
url = "https://raw.githubusercontent.com/elchapa24/tpi-analisis-de-datos/main/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras 5 filas:")
display(df.head())

print("\nInformación de tipos de datos ANTES de limpiar:")
df.info()

Dimensiones del dataset: (7043, 21)

Primeras 5 filas:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Información de tipos de datos ANTES de limpiar:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object

In [2]:
# 2. LIMPIEZA DE DATOS
# Convertimos TotalCharges a numerico.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')

# Verificamos nulos generados
nulos_total = df["TotalCharges"].isnull().sum()
print(f"Valores nulos detectados en TotalCharges: {nulos_total}")

# Imputamos con 0 (clientes nuevos sin historial de pago)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Eliminamos customerID (no aporta)
df_clean = df.drop(columns=["customerID"])

# Verificamos tipos de datos limpios
print("\nTipos de datos DESPUÉS de limpiar:")
print(df_clean.dtypes)

Valores nulos detectados en TotalCharges: 11

Tipos de datos DESPUÉS de limpiar:
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


In [3]:
# 3. DETECCION DE OUTLIERS
def calcular_outliers_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

columnas_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in columnas_numericas:
    li, ls = calcular_outliers_iqr(df_clean[col])
    outliers = df_clean[(df_clean[col] < li) | (df_clean[col] > ls)]
    print(f"Variable {col}: Límite Inf {li:.2f} | Límite Sup {ls:.2f} | Outliers: {len(outliers)}")

# Decision: Conservamos los outliers porque representan clientes reales de alto valor

# 4. CREACION DE COLUMNA CALCULADA
df_clean["CargoDiario"] = df_clean["MonthlyCharges"] / 30
print("\n✅ Dataset limpio y transformado.")
display(df_clean[["MonthlyCharges", "CargoDiario", "Churn"]].head())

Variable tenure: Límite Inf -60.00 | Límite Sup 124.00 | Outliers: 0
Variable MonthlyCharges: Límite Inf -46.02 | Límite Sup 171.38 | Outliers: 0
Variable TotalCharges: Límite Inf -4683.52 | Límite Sup 8868.67 | Outliers: 0

✅ Dataset limpio y transformado.


,MonthlyCharges,CargoDiario,Churn
0,29.85,0.995000,No
1,56.95,1.898333,No
2,53.85,1.795000,Yes
3,42.30,1.410000,No
4,70.70,2.356667,Yes
